In [49]:
import os
import pandas as pd
import json
import random
import sys
sys.path.append('C:/Users/YangTeng/Desktop/venv/00.module')
import UA
import Proxy

# = = = = = = = = = = = = = = = = = =

from selenium.webdriver import ChromeOptions
option = ChromeOptions()
option.add_extension(Proxy.get_Proxy_Selenium())
option.add_experimental_option('useAutomationExtension', False)
option.add_argument('blink-settings=imagesEnabled=false')
option.add_argument('--disable-blink-features=AutomationControlled')
option.add_argument('--disable-dev-shm-usage')
option.add_argument('--disable-gpu')
option.add_experimental_option('excludeSwitches', ['enable-automation'])
# option.add_argument('--headless')
option.add_argument('--no-sandbox')
option.add_argument(UA.get_User_Agent_Selenium())
option.page_load_strategy = 'eager'

from selenium.webdriver import Chrome
from selenium.webdriver.chrome.service import Service
import time
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.by import By


df_menu = pd.read_excel('C:/Users/YangTeng/Desktop/爬虫.xlsx', header=0)
list_OE = df_menu['OE'].to_list()
list_OE


['GS251220', 'GS251221', 'GS501165', 'CMS501275', 'CMS501276', 'CMS30103']

In [50]:

df = pd.DataFrame(columns=['Part_Number',
                           'Manufacturer',
                           'Vehicle',
                           'Note_1',
                           'Info',
                           'Note_2',
                           'Alternate_OE_Part_Numbers',
                           'Src',
                           'status'])

df

,Part_Number,Manufacturer,Vehicle,Note_1,Info,Note_2,Alternate_OE_Part_Numbers,Src,status


In [51]:
browser = Chrome(service=Service(r'C:\Users\YangTeng\Desktop\venv\chromedriver.exe'), options=option)

browser.maximize_window()

In [53]:
for i in list_OE[0:len(list_OE)]:
    print(i)
    a = str(i)
    
    browser.get('https://www.rockauto.com/en/partsearch/?partnum='+str(i))

    time.sleep(3)

    wait = WebDriverWait(browser, 60)    

#     wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="partnum_partsearch_007"]')))

#     input_OE = browser.find_element(by=By.XPATH, value='//*[@id="partnum_partsearch_007"]')

#     input_OE.clear()

#     # a = str(int(Dorman[i]))

#     input_OE.send_keys(a,Keys.ENTER)

#     time.sleep(5)

    url = browser.current_url
    if len(browser.find_elements(by=By.XPATH, value='//span[@class="navlabellink nvoffset nnormal"]')) != 0:
        df_temp = pd.DataFrame([{'Part_Number': a,
                                'status' : 'No Parts Found' }])
        continue

    wait.until(EC.presence_of_element_located((By.XPATH, '//tbody[@class="listing-inner altrow-a-1  "]')))

    list_part = browser.find_elements(by=By.XPATH, value='//tbody[@class="listing-inner altrow-a-1  "]') + browser.find_elements(by=By.XPATH, value='//tbody[@class="listing-inner altrow-a-0  "]')

    list_part

    list_part_number = [part.find_element(by=By.XPATH, value='./descendant::div[@class="listing-text-row-moreinfo-truck"]/span[@title="Buyer\'s Guide"]') for part in list_part]

    list_part_number

    part_number_text = [part_number.text for part_number in list_part_number]
    
    if a in part_number_text:
        part_index = part_number_text.index(a)

        manufacturer = list_part[part_index].find_element(by=By.XPATH, value='./descendant::div[@class="listing-text-row-moreinfo-truck"]/span[@class="listing-final-manufacturer "]').text

        manufacturer

        list_part_number[part_index].click()
        time.sleep(5)
        list_vehicle = browser.find_element(by=By.XPATH, value='//div[@id="buyersguidepopup-outer_c"]').find_elements(by=By.XPATH, value='./descendant::table[@class="nobmp"]')
        if len(list_vehicle) == 0:
            vehicle = ''
        else:
            vehicle = list_vehicle[0].text

        vehicle
        note_1 = ';'.join(row.text for row in list_part[part_index].find_elements(by=By.XPATH, value='./descendant::div[@class="listing-text-row-moreinfo-truck"]/span[@class="span-link-underline-remover"]'))

        note_1

        info = ';'.join(row.get_attribute('href') for row in list_part[part_index].find_elements(by=By.XPATH, value='./descendant::div[@class="listing-text-row-moreinfo-truck"]/a[@class="ra-btn ra-btn-moreinfo"]'))

        info

        note_2 = ';'.join(row.text for row in list_part[part_index].find_elements(by=By.XPATH, value='./descendant::div[@class="listing-text-row"]'))

        note_2

        oe = ';'.join(row.get_attribute('textContent') for row in list_part[part_index].find_elements(by=By.XPATH, value='./descendant::span[@title="Replaces these Alternate/ OE Part Numbers"]'))
        oe = ';'.join(oe.split(', '))

        oe

        src = ';'.join(row.get_attribute('src') for row in list_part[part_index].find_elements(by=By.XPATH, value='./descendant::img[@class=" listing-inline-image"]'))

        src

        df_temp = pd.DataFrame([{'Part_Number': a,
                             'Manufacturer': manufacturer,
                             'Vehicle': vehicle,
                             'Note_1': note_1,
                             'Info': info,
                             'Note_2': note_2,
                             'Alternate_OE_Part_Numbers': oe,
                             'Src': src,
                             'url':url,
                             'status': 'ok'}])
    else:
        df_temp = pd.DataFrame([{'Part_Number': a,
                                'status' : '手工' }])
        


    df = pd.concat([df, df_temp], ignore_index=True).fillna('')
    print(df)
df

GS251220
GS251221
GS501165
CMS501275
  Part_Number Manufacturer Vehicle Note_1 Info Note_2  \
0   CMS501275                                           

  Alternate_OE_Part_Numbers Src status  
0                                   手工  
CMS501276
  Part_Number Manufacturer Vehicle Note_1 Info Note_2  \
0   CMS501275                                           
1   CMS501276                                           

  Alternate_OE_Part_Numbers Src status  
0                                   手工  
1                                   手工  
CMS30103
  Part_Number Manufacturer                                           Vehicle  \
0   CMS501275                                                                  
1   CMS501276                                                                  
2    CMS30103     MEVOTECH  INFINITI FX35 2003-2008\nINFINITI FX45 2003-2008   

                                              Note_1  \
0                                                      
1                  

,Part_Number,Manufacturer,Vehicle,Note_1,Info,Note_2,Alternate_OE_Part_Numbers,Src,status,url
0,CMS501275,,,,,,,,手工,
1,CMS501276,,,,,,,,手工,
2,CMS30103,MEVOTECH,INFINITI FX35 2003-2008\nINFINITI FX45 2003-2008,Supreme; Complete Arm w/ Ball Joint; Includes ...,https://www.rockauto.com/en/moreinfo.php?pk=61...,Category: Control Arm,55502CG000;MS30103,https://www.rockauto.com/info/143/MS30103_FRO_...,ok,https://www.rockauto.com/en/partsearch/?partnu...


In [54]:
df

,Part_Number,Manufacturer,Vehicle,Note_1,Info,Note_2,Alternate_OE_Part_Numbers,Src,status,url
0,CMS501275,,,,,,,,手工,
1,CMS501276,,,,,,,,手工,
2,CMS30103,MEVOTECH,INFINITI FX35 2003-2008\nINFINITI FX45 2003-2008,Supreme; Complete Arm w/ Ball Joint; Includes ...,https://www.rockauto.com/en/moreinfo.php?pk=61...,Category: Control Arm,55502CG000;MS30103,https://www.rockauto.com/info/143/MS30103_FRO_...,ok,https://www.rockauto.com/en/partsearch/?partnu...


In [10]:
list_OE[142]

'CMS401283'

In [55]:
df.to_excel('C:/Users/YangTeng/Desktop/爬虫-补充.xlsx', index=False)

In [ ]:
384+